Bohua Liu is responsible for this code

**This code applied the idea of offline augmentation, but it is not finished due to the insufficient VM hours**

### Offline Augmentation (see appendix in report for detailed explaination)

In [ ]:
import os
import cv2
import numpy as np
import random
from tqdm import tqdm
import shutil

# CONFIGURATION
# Input folder (Your current segmented dataset)
INPUT_ROOT = "dataset_yolo_4cats_seg"

# Output folder (Where the new Augmented data will go)
OUTPUT_ROOT = "yolo_4cats_seg_augmented"

# THE STRATEGY:
# For every specific Glass object found in Train, paste it into 5 OTHER Train images.
# For every specific Paper object found in Train, paste it into 3 OTHER Train images.
COPIES_PER_OBJECT = {
    1: 5,  # Glass
    2: 3   # Paper
}

# SAFETY LIMITS
MAX_PASTE_PER_IMAGE = 5   # Don't put more than 4 extra items on one image
MIN_OBJ_SCALE = 0.01      # Sticker must be at least 5% of image height
MAX_OBJ_SCALE = 0.20      # Sticker must be at most 20% of image height

def load_yolo_polygon(txt_path, w, h):
    """Parses YOLO txt labels into a list of dicts with polygons and bboxes."""
    objects = []
    if not os.path.exists(txt_path): return objects

    with open(txt_path, 'r') as f:
        for line in f.readlines():
            parts = list(map(float, line.strip().split()))
            cls = int(parts[0])
            coords = parts[1:]
            # Denormalize points
            pts = np.array([(coords[i]*w, coords[i+1]*h) for i in range(0, len(coords), 2)], dtype=np.int32)
            # Calculate bbox for fast collision checking
            x, y, bw, bh = cv2.boundingRect(pts)
            objects.append({'cls': cls, 'points': pts, 'bbox': (x, y, x+bw, y+bh)})
    return objects

def check_overlap(new_box, existing_objects):
    """Returns True if new_box overlaps with any existing object."""
    nx, ny, nx2, ny2 = new_box
    for obj in existing_objects:
        ox, oy, ox2, oy2 = obj['bbox']

        # Intersection Box
        xx1 = max(nx, ox)
        yy1 = max(ny, oy)
        xx2 = min(nx2, ox2)
        yy2 = min(ny2, oy2)

        w = max(0, xx2 - xx1)
        h = max(0, yy2 - yy1)

        # If intersection area > 0, it's an overlap
        if w * h > 0:
            return True
    return False

def extract_minority_objects_from_train():
    """Scans ONLY the TRAIN folder to harvest Glass and Paper objects."""
    img_dir = os.path.join(INPUT_ROOT, "images", "train")
    lbl_dir = os.path.join(INPUT_ROOT, "labels", "train")

    extracted = {1: [], 2: []}

    files = [f for f in os.listdir(img_dir) if f.endswith('.jpg')]
    print(f"--- Scanning {len(files)} Train images for Glass/Paper ---")

    for fname in tqdm(files):
        lbl_path = os.path.join(lbl_dir, fname.replace('.jpg', '.txt'))
        if not os.path.exists(lbl_path): continue

        # Quick check if label contains target classes before loading image
        has_target = False
        with open(lbl_path, 'r') as f:
            for line in f:
                if int(line.split()[0]) in COPIES_PER_OBJECT:
                    has_target = True; break
        if not has_target: continue

        # Load
        img = cv2.imread(os.path.join(img_dir, fname))
        if img is None: continue
        h, w = img.shape[:2]

        objs = load_yolo_polygon(lbl_path, w, h)

        for obj in objs:
            cls = obj['cls']
            if cls in extracted:
                # Create Mask
                mask = np.zeros((h, w), dtype=np.uint8)
                cv2.fillPoly(mask, [obj['points']], 255)

                # Crop
                x, y, bw, bh = obj['bbox'][0], obj['bbox'][1], obj['bbox'][2]-obj['bbox'][0], obj['bbox'][3]-obj['bbox'][1]

                # Filter tiny noise
                if bw > 10 and bh > 10:
                    crop_img = img[y:y+bh, x:x+bw]
                    crop_mask = mask[y:y+bh, x:x+bw]
                    # Points relative to the crop
                    local_points = obj['points'] - np.array([x, y])

                    extracted[cls].append({
                        'img': crop_img,
                        'mask': crop_mask,
                        'poly': local_points
                    })
    return extracted


# 0. Cleanup Output
if os.path.exists(OUTPUT_ROOT): shutil.rmtree(OUTPUT_ROOT)
for subset in ['train', 'val']:
    os.makedirs(os.path.join(OUTPUT_ROOT, "images", subset), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_ROOT, "labels", subset), exist_ok=True)

# 1. Harvest Objects (FROM TRAIN ONLY)
minority_objs = extract_minority_objects_from_train()
print(f"Harvested: {len(minority_objs[1])} Glass objects, {len(minority_objs[2])} Paper objects.")

# 2. Build Schedule (FOR TRAIN ONLY)
train_files = [f for f in os.listdir(os.path.join(INPUT_ROOT, "images", "train")) if f.endswith('.jpg')]
paste_schedule = {f: [] for f in train_files}

print("--- Scheduling Distribution ---")
for cls_id, obj_list in minority_objs.items():
    copies_needed = COPIES_PER_OBJECT[cls_id]

    for obj_data in obj_list:
        # Pick N random training images to host this object
        targets = random.sample(train_files, copies_needed)
        for t_img in targets:
            if len(paste_schedule[t_img]) < MAX_PASTE_PER_IMAGE:
                # Queue tuple: (class_id, object_data)
                paste_schedule[t_img].append((cls_id, obj_data))

# 3. Execute Augmentation (ON TRAIN SET)
print("--- Processing TRAIN Set (Augmenting) ---")
src_img_dir = os.path.join(INPUT_ROOT, "images", "train")
src_lbl_dir = os.path.join(INPUT_ROOT, "labels", "train")
dst_img_dir = os.path.join(OUTPUT_ROOT, "images", "train")
dst_lbl_dir = os.path.join(OUTPUT_ROOT, "labels", "train")

for fname in tqdm(train_files):
    img_path = os.path.join(src_img_dir, fname)
    lbl_path = os.path.join(src_lbl_dir, fname.replace('.jpg', '.txt'))

    img = cv2.imread(img_path)
    h_bg, w_bg = img.shape[:2]

    # Load existing labels
    current_labels = load_yolo_polygon(lbl_path, w_bg, h_bg)

    # Check schedule
    items_to_paste = paste_schedule[fname]

    for item in items_to_paste:
        cls_id, obj_data = item
        s_img = obj_data['img']
        s_mask = obj_data['mask']
        s_poly = obj_data['poly']

        # Random Scaling (Relative to Background Height)
        target_h = random.uniform(MIN_OBJ_SCALE, MAX_OBJ_SCALE) * h_bg
        scale_factor = target_h / s_img.shape[0]

        new_w = int(s_img.shape[1] * scale_factor)
        new_h = int(s_img.shape[0] * scale_factor)

        s_img_res = cv2.resize(s_img, (new_w, new_h))
        s_mask_res = cv2.resize(s_mask, (new_w, new_h))
        s_poly_res = (s_poly * scale_factor).astype(np.int32)

        # Collision Check (Try 10 random spots)
        for _ in range(10):
            if w_bg - new_w <= 0 or h_bg - new_h <= 0: break

            paste_x = random.randint(0, w_bg - new_w)
            paste_y = random.randint(0, h_bg - new_h)
            new_bbox = (paste_x, paste_y, paste_x+new_w, paste_y+new_h)

            if not check_overlap(new_bbox, current_labels):
                # Paste Pixels
                roi = img[paste_y:paste_y+new_h, paste_x:paste_x+new_w]
                mask_3ch = cv2.merge([s_mask_res, s_mask_res, s_mask_res])
                img_paste = np.where(mask_3ch > 127, s_img_res, roi)
                img[paste_y:paste_y+new_h, paste_x:paste_x+new_w] = img_paste

                # Update Label List
                new_points = s_poly_res + np.array([paste_x, paste_y])
                current_labels.append({'cls': cls_id, 'points': new_points, 'bbox': new_bbox})
                break

    # Save Final Train Image
    cv2.imwrite(os.path.join(dst_img_dir, fname), img)

    # Save Final Train Label
    with open(os.path.join(dst_lbl_dir, fname.replace('.jpg', '.txt')), 'w') as f:
        for obj in current_labels:
            norm_pts = []
            for pt in obj['points']:
                nx = max(0, min(1, pt[0] / w_bg))
                ny = max(0, min(1, pt[1] / h_bg))
                norm_pts.extend([nx, ny])
            f.write(f"{obj['cls']} " + " ".join(map(str, norm_pts)) + "\n")

# 4. Handle Validation (STRICT COPY - NO MODIFICATION)
print("--- Processing VAL Set (Strict Copy Only) ---")
val_files = [f for f in os.listdir(os.path.join(INPUT_ROOT, "images", "val")) if f.endswith('.jpg')]

for fname in tqdm(val_files):
    # Paths
    src_img = os.path.join(INPUT_ROOT, "images", "val", fname)
    src_txt = os.path.join(INPUT_ROOT, "labels", "val", fname.replace('.jpg', '.txt'))

    dst_img = os.path.join(OUTPUT_ROOT, "images", "val", fname)
    dst_txt = os.path.join(OUTPUT_ROOT, "labels", "val", fname.replace('.jpg', '.txt'))

    # Copy Image
    shutil.copy(src_img, dst_img)

    # Copy Label (if it exists)
    if os.path.exists(src_txt):
        shutil.copy(src_txt, dst_txt)

print(f"\nDone! Augmented dataset saved to: {OUTPUT_ROOT}")


--- Scanning 1275 Train images for Glass/Paper ---


100%|██████████| 1275/1275 [00:22<00:00, 55.82it/s]


Harvested: 234 Glass objects, 435 Paper objects.
--- Scheduling Distribution ---
--- Processing TRAIN Set (Augmenting) ---


100%|██████████| 1275/1275 [02:52<00:00,  7.39it/s]


--- Processing VAL Set (Strict Copy Only) ---


100%|██████████| 225/225 [00:01<00:00, 182.14it/s]


Done! Augmented dataset saved to: yolo_4cats_seg_augmented


### Image Tiling

In [ ]:
# Slicing Script (Updated for Train AND Val)
import os
import cv2
import numpy as np
from shapely.geometry import Polygon, box
from tqdm import tqdm
import shutil

# CONFIGURATION
# ROOT INPUT: The folder containing 'train' and 'val' folders
INPUT_ROOT = "yolo_4cats_seg_augmented"

# ROOT OUTPUT: Where the new tiled dataset will be created
OUTPUT_ROOT = "yolo_4cats_seg_aug_tiled"

TILE_SIZE = 640
OVERLAP = 0.2
MIN_VISIBILITY = 0.5

def slice_subset(subset_name):
    """
    Slices a specific subset (e.g., 'train' or 'val')
    """
    input_img_dir = os.path.join(INPUT_ROOT, "images", subset_name)
    input_lbl_dir = os.path.join(INPUT_ROOT, "labels", subset_name)

    output_img_dir = os.path.join(OUTPUT_ROOT, "images", subset_name)
    output_lbl_dir = os.path.join(OUTPUT_ROOT, "labels", subset_name)

    # Create directories
    os.makedirs(output_img_dir, exist_ok=True)
    os.makedirs(output_lbl_dir, exist_ok=True)

    # Check if input directory exists
    if not os.path.exists(input_img_dir):
        print(f"Warning: {input_img_dir} not found. Skipping.")
        return

    img_files = [f for f in os.listdir(input_img_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]
    print(f"--- Processing '{subset_name}' set ({len(img_files)} images) ---")

    tile_count = 0

    for img_name in tqdm(img_files):
        img_path = os.path.join(input_img_dir, img_name)
        lbl_path = os.path.join(input_lbl_dir, os.path.splitext(img_name)[0] + ".txt")

        img = cv2.imread(img_path)
        if img is None: continue
        h_img, w_img, _ = img.shape

        # Load Polygons
        polygons = []
        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as f:
                for line in f.readlines():
                    parts = list(map(float, line.strip().split()))
                    cls = int(parts[0])
                    coords = parts[1:]
                    pts = [(coords[i]*w_img, coords[i+1]*h_img) for i in range(0, len(coords), 2)]
                    if len(pts) >= 3:
                        poly = Polygon(pts)
                        if poly.is_valid:
                            polygons.append({'cls': cls, 'poly': poly})

        # Grid Calculation
        step = int(TILE_SIZE * (1 - OVERLAP))

        # Slicing Loop
        for y in range(0, h_img, step):
            for x in range(0, w_img, step):
                x_start = min(x, w_img - TILE_SIZE)
                y_start = min(y, h_img - TILE_SIZE)
                if x_start < 0: x_start = 0
                if y_start < 0: y_start = 0
                x_end, y_end = x_start + TILE_SIZE, y_start + TILE_SIZE

                tile_box = box(x_start, y_start, x_end, y_end)
                new_labels = []

                # Check Intersections
                for obj in polygons:
                    if tile_box.intersects(obj['poly']):
                        inter = tile_box.intersection(obj['poly'])
                        if inter.area / obj['poly'].area > MIN_VISIBILITY:
                            if inter.geom_type == 'Polygon':
                                coords = list(inter.exterior.coords)
                            elif inter.geom_type == 'MultiPolygon':
                                coords = list(max(inter.geoms, key=lambda a: a.area).exterior.coords)
                            else: continue

                            norm_coords = []
                            for pt in coords:
                                nx = max(0, min(1, (pt[0] - x_start) / TILE_SIZE))
                                ny = max(0, min(1, (pt[1] - y_start) / TILE_SIZE))
                                norm_coords.extend([nx, ny])

                            new_labels.append(f"{obj['cls']} {' '.join(map(str, norm_coords))}")

                # Save only non-empty tiles
                if len(new_labels) > 0:
                    tile_name = f"{os.path.splitext(img_name)[0]}_{x_start}_{y_start}"
                    cv2.imwrite(os.path.join(output_img_dir, tile_name + ".jpg"), img[y_start:y_end, x_start:x_end])
                    with open(os.path.join(output_lbl_dir, tile_name + ".txt"), 'w') as f:
                        f.write('\n'.join(new_labels))
                    tile_count += 1

    print(f"Finished '{subset_name}'. Generated {tile_count} dense tiles.")

# EXECUTION
if os.path.exists(OUTPUT_ROOT): shutil.rmtree(OUTPUT_ROOT) # Clean start
slice_subset("train")
slice_subset("val")
print(f"All done! Dataset saved to {OUTPUT_ROOT}")


--- Processing 'train' set (1275 images) ---


100%|██████████| 1275/1275 [02:57<00:00,  7.18it/s]


Finished 'train'. Generated 8310 dense tiles.
--- Processing 'val' set (225 images) ---


100%|██████████| 225/225 [00:14<00:00, 15.38it/s]


Finished 'val'. Generated 798 dense tiles.
All done! Dataset saved to yolo_4cats_seg_aug_tiled


FileNotFoundError: [Errno 2] No such file or directory: 'dataset_yolo_tiled/taco_tiled.yaml'

Create yaml file

In [ ]:
import os
from ultralytics import YOLO

# Create the config file pointing to the NEW sliced folders
tiled_yaml_content = f"""
path: {os.path.abspath("yolo_4cats_seg_aug_tiled")}
train: images/train
val: images/val
names:
  0: plastic
  1: glass
  2: paper
  3: unsorted
"""

with open("yolo_4cats_seg_aug_tiled/taco.yaml", "w") as f:
    f.write(tiled_yaml_content)

In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 32.8 MB/s eta 0:00:00


Train Model

In [ ]:
import os
from ultralytics import YOLO

# 1. USE PRE-TRAINED WEIGHTS (The smart starting point)
model = YOLO('yolov8s.pt')

# 2. TRAIN WITH "HARD MODE" SETTINGS
model.train(
    data="yolo_4cats_seg_aug_tiled/taco.yaml",
    epochs=200,
    batch=32,
    imgsz=640,
    name="yolov8s_aug_tiled",
    project="Yolo_training",
    seed=42,

    # HARD AUGMENTATION
    degrees=15.0,    # Rotate slightly
    flipud=0.5,      # Flip upside down (Trash has no orientation)
    fliplr=0.5,      # Flip left right
    scale=0.2,       # Reduced

    mosaic=1.0,      # Always use Mosaic (4 images in 1)
    mixup=0.0,       # VITAL: Don't blend trash with background
    copy_paste=0.1,  # Partially done offline
    erasing=0.0,   # Set to 0. DO NOT erase parts of small objects.

    # --- COLOR VARIATION ---
    hsv_h=0.015,     # Keep hue low (don't change red to green)
    hsv_s=0.5,       # Saturation variation
    hsv_v=0.4,       # Brightness variation (Day/Night sim)

    # --- OPTIMIZATION ---
    lr0=0.01,        # Standard starting LR
    lrf=0.01,        # Final LR (divides lr0 by 100)
    patience=30,     # Give it time to struggle and recover
    close_mosaic=10, # Turn off the "hard" mosaic effects for the last 20 epochs

    # 1. BOOST CLASS LOSS (The "Don't Ignore Me" Knob)
    # Default is usually 0.5. Raising this forces the model to pay
    # extreme attention to faint signals of glass/paper.
    cls=3.5,
    # 2. LOWER BOX LOSS
    # Default is 7.5. Lowering this tells the model:
    # "Focus on finding the object (cls) first, refine the shape (box) later."
    box=5.5,

    # --- REGULARIZATION ---
    # This prevents the model from memorizing the tiles
    weight_decay=0.0005,
    label_smoothing=0.1,
)

WARNING ⚠️ 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.3.235 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=5.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=3.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_4cats_seg_aug_tiled/taco.yaml, degrees=15.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.0, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.5, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8s_aug_tiled2, nbs=64, nms=False, opset=None, o

In [ ]:
from ultralytics import YOLO
import os
os.environ["WANDB_DISABLED"] = "true"

# Load the last saved model
model = YOLO("Yolo_training/yolov8s_aug_tiled2/weights/last.pt")

# Resume training
model.train(
    resume=True,
)


Ultralytics 8.3.235 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=5.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=3.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_4cats_seg_aug_tiled/taco.yaml, degrees=15.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.0, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.5, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=Yolo_training/yolov8s_aug_tiled2/weights/last.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8s_aug_tiled2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_m